# 1D J1-J2=0.0-J3=0.5: EuclRNN & PoincareRNN

This is part of the work https://arxiv.org/abs/2604.24337. In this notebook we performed 2 kinds of trainings with Euclidean RNN and Poincare RNN. 
- The first kind involves training without the temperature scaling `tau' in the `sample' method.
- The second kind involves training with `tau'.
The results do not seem to differ qualitatively with or without tau. 

In [1]:
import os
import sys
import time
sys.path.append('../utility')
#from j1j2j3_hyprnn_train_loop import *
#for training using sample generation involving the temp scaling tau:
from j1j2j3_tau_hyprnn_train_loop import *

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

In [3]:
E_exact = -53.99140745384453

syssize = 100
nssamples = 80
J1 = 1.0
J2 = 0.0
J3 = 0.5
nsteps = 1201
var_tol = 20.0

# euclRNN

In [4]:
#with tau: tau seems to have no effect whatsoever on the convergence of eRNN to a false mininum
units = 80
fname=f'1d_J1J2J3_results_N=100_tau/eRNN/J2={J2}_J3={J3}'
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)

for name, param in eRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params = {sum(p.numel() for p in eRNN.model.parameters())}')

t0=time.time()

mE, vE = run_J1J2J3(eRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([80, 80])
Layer: rnn.U | Size: torch.Size([2, 80])
Layer: rnn.b | Size: torch.Size([1, 80])
Layer: dense_a.weight | Size: torch.Size([2, 80])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 80])
Layer: dense_p.bias | Size: torch.Size([2])
Total params = 6964


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.23145, mean energy: 36.79937-0.00410j, varE: 0.31930, |LR: 5.00e-03|tau=1.05
Best model saved at epoch 1 with best E=36.36862-0.06659j, varE=2.32178
step: 10, loss: -18.63008, mean energy: -29.96698+0.09446j, varE: 52.90653, |LR: 5.00e-03|tau=1.0490000000000002
Best model saved at epoch 11 with best E=-33.39928-0.00066j, varE=7.40264
Best model saved at epoch 12 with best E=-34.78471+0.05454j, varE=5.16881
Best model saved at epoch 13 with best E=-35.59312+0.05298j, varE=6.13038
Best model saved at epoch 14 with best E=-35.93307-0.04232j, varE=2.52692
Best model saved at epoch 16 with best E=-36.27596-0.00336j, varE=1.59125
Best model saved at epoch 17 with best E=-36.46115+0.02883j, varE=1.80548
Best model saved at epoch 18 with best E=-36.50928+0.01356j, varE=1.54332
Best model saved at epoch 19 with best E=-36.81131+0.00967j, varE=1.27601
step: 20, loss: -1.29897, mean energy: -36.67381-0.04587j, varE: 1.78344, |LR: 5.00e-03|tau=1.048
Best model saved at epoch 21 w

In [7]:
units = 80
fname=f'1d_J1J2J3_results_N={syssize}/eRNN/J2={J2}_J3={J3}'
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)

for name, param in eRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params = {sum(p.numel() for p in eRNN.model.parameters())}')

t0=time.time()

mE, vE = run_J1J2J3(eRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.42212, mean energy: 36.84056-0.00380j, varE: 0.43467, |LR: 5.00e-03
Best model saved at epoch 1 with best E=36.38049-0.05049j, varE=2.38649
step: 10, loss: 1.50028, mean energy: 0.08588+0.45033j, varE: 62.04471, |LR: 5.00e-03
Best model saved at epoch 19 with best E=-33.73030-0.07749j, varE=14.55438
Best model saved at epoch 20 with best E=-35.30123+0.03866j, varE=7.11126
step: 20, loss: -7.00002, mean energy: -35.30123+0.03866j, varE: 7.11126, |LR: 5.00e-03
Best model saved at epoch 21 with best E=-36.30259+0.02611j, varE=3.63533
Best model saved at epoch 22 with best E=-36.97026-0.04086j, varE=2.24824
Best model saved at epoch 29 with best E=-36.98464-0.00453j, varE=1.35747
Best model saved at epoch 30 with best E=-37.25338+0.00614j, varE=0.78065
step: 30, loss: -1.02381, mean energy: -37.25338+0.00614j, varE: 0.78065, |LR: 5.00e-03
Best model saved at epoch 36 with best E=-37.31438+0.00035j, varE=0.52509
Best model saved at epoch 37 with best E=-37.35354+0.00194j, 

# Poincare RNN

### No tau scaling in sample generation

In [9]:
r_max = 0.78
units = 80
fname=f'1d_J1J2J3_results_N={syssize}/hRNN_rmax={r_max}/J2={J2}_J3={J3}'
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN',r_max = r_max,  bias_geom='hyp',
                           hyp_non_lin='id', units=units, seed=111)
for name, param in hRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params = {sum(p.numel() for p in hRNN.model.parameters())}')

Layer: rnn.W | Size: torch.Size([80, 80])
Layer: rnn.U | Size: torch.Size([2, 80])
Layer: rnn.b | Size: torch.Size([1, 80])
Layer: dense_a.weight | Size: torch.Size([2, 80])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 80])
Layer: dense_p.bias | Size: torch.Size([2])
Total params = 6964


In [10]:
t0=time.time()
mhE, vhE = run_J1J2J3(hRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

step: 0, loss: -1.42188, mean energy: 36.84056-0.00380j, varE: 0.43467| Max Radius: 0.0146 | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=36.39909-0.05748j, varE=2.29171
step: 10, loss: 11.93718, mean energy: 0.81315+0.51642j, varE: 30.71607| Max Radius: 0.4574 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 20, loss: -7.46631, mean energy: -11.38475-0.52916j, varE: 39.81808| Max Radius: 0.7760 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 30, loss: -14.14563, mean energy: -35.19107-0.05033j, varE: 37.03693| Max Radius: 0.9285 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 40, loss: -13.24817, mean energy: -38.59128-0.08148j, varE: 25.38905| Max Radius: 0.9305 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 50, loss: 13.39978, mean energy: -41.53893-0.17505j, varE: 25.73650| Max Radius: 0.9398 | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 52 with best E=-42.09715+0.05449j, varE=17.42900
Best model saved at epoch 56 with best E=-44.20264-0.14558j, varE=18.17968
Best model saved at

### With tau scaling in sample generation

In [10]:
r_max = 0.78
units = 80
fname=f'1d_J1J2J3_results_N={syssize}/hRNN_tau_rmax={r_max}/J2={J2}_J3={J3}'
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN',r_max = r_max,  bias_geom='hyp',
                           hyp_non_lin='id', units=units, seed=111)
for name, param in hRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params = {sum(p.numel() for p in hRNN.model.parameters())}')

t0=time.time()
mhE, vhE = run_J1J2J3(hRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([80, 80])
Layer: rnn.U | Size: torch.Size([2, 80])
Layer: rnn.b | Size: torch.Size([1, 80])
Layer: dense_a.weight | Size: torch.Size([2, 80])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 80])
Layer: dense_p.bias | Size: torch.Size([2])
Total params = 6964


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.23120, mean energy: 36.79937-0.00410j, varE: 0.31930| Max Radius: 0.0146 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.05
Best model saved at epoch 1 with best E=36.42552-0.04195j, varE=2.30223
step: 10, loss: 124.45581, mean energy: 0.54692-1.24760j, varE: 64.56344| Max Radius: 0.8824 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.0490000000000002
step: 20, loss: 8.87524, mean energy: -23.57658+0.24686j, varE: 63.69186| Max Radius: 0.8935 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.048
step: 30, loss: -26.73694, mean energy: -33.21622+0.25086j, varE: 45.03406| Max Radius: 0.8749 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.0470000000000002
step: 40, loss: 11.41772, mean energy: -37.61421-0.06104j, varE: 59.25080| Max Radius: 0.8775 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.046
step: 50, loss: -3.69067, mean energy: -42.79211+0.05155j, varE: 30.35776| Max Radius: 0.9068 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.0450000000000002
Best model saved at epoch 59 with best E=-44.40325+0.04823j, varE=18.735